# Qdrant Hybrid Search

- Qdrant Vector DB는 `sparse`와 `dense` vector의 검색 결과를 결합한 hybrid search를 지원.

- `dense`는 앞서 계속 봐왔던 vector들. text 전체에 걸쳐 풍부한 semantics를 포착할 수 있음
- `sparse`는 TF-IDF, BM25와 같은 특수한 approach를 사용해서 생성됨.
  - 특정 키워드와 유사한 small detail을 포착하는데 효과적.


## Setup


In [ ]:
from rich import print # 이쁜 output
from importlib.metadata import version

packages = ["google-genai",
            "llama-index",
            "llama-index-vector-stores-qdrant",
            "llama-index-readers-file",
            "llama-index-embeddings-fastembed",
            "llama-index-llms-google-genai",
            "qdrant_client",
            "fastembed",
            ]

for package in packages:
    try:
        pkg_version = version(package)
    except Exception as e:
        pkg_version = f"Error retrieving version: {e}"
    print(f"{package}: {pkg_version}")

In [ ]:
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
# !wget --user-agent "Mozilla" "https://arxiv.org/pdf/2307.09288.pdf" -O "rag_dataset/llama2.pdf"

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./rag_dataset/").load_data()

example 에선 wget으로 하는데, encoding error인지 뭔지가 나와서 그냥 다운로드 받음.


## Indexing data

- data가 들어왔으니 이제 indexing 할 차례.
- Qdrant를 사용한 hybrid search는 처음부터 설정을 해줘야 함.
  - 단순히 `enable_hybrid=True`로 하면 됨.
- 이렇게 하면 지정한 LLM을 사용해 dense vector를 생성하는 것 외에도, fastembed의 `prithvida/Splade_PP_en_v1`를 사용해서 sparse vector도 생성함.


In [ ]:
from httpx import Timeout

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, AsyncQdrantClient

# OpenAI -> Gemini
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.fastembed import FastEmbedEmbedding

# default로 OpenAI embedding을 사용하니, Setting으로 바꿔주기.
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = GoogleGenAI(
    model = 'gemini-2.5-flash',
    api_key = GEMINI_API_KEY
)

# disk에 persistent index 생성
# client = QdrantClient(host='localhost', port=6333)
# aclient = AsyncQdrantClient(host='localhost', port=6333)
client = QdrantClient(path='./qdrant_db')

# hybrid indexing이 활성화 된 vector DB를 생성
# batch_size는 sparse vector를 사용해서 한번에 encoding 되는 node의 수를 제어
vector_store = QdrantVectorStore(
    collection_name = 'llama2_paper',
    client = client,
    enable_hybrid = True,
    batch_size = 20
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
Settings.chunk_size = 512

index = VectorStoreIndex.from_documents(
    documents = documents,
    storage_context = storage_context
)

vector DB 생성에 무슨 30분이 넘게 걸린다 생성도 안되고...

[example repo](https://github.com/adithya-s-k/AI-Engineering.academy/blob/main/docs/RAG/03_Hybrid_RAG/qdrant_hybrid.ipynb) review로 진행.
